# TxGNN KG Schema — Overview & Usage Guide

> Refreshed for relation coverage audit `t_0b1f53d9` on 2026-06-23. The current source-of-truth coverage table is `docs/relation_coverage_current.md`; this notebook is executable schema/API documentation, not the canonical coverage ledger.

This notebook walks through all elements of `manage_db/kg_schema.py`:

1. [Node types & primary ontologies](#1-node-types)
2. [Cross-reference columns](#2-cross-reference-columns)
3. [Relation taxonomy](#3-relation-taxonomy)
4. [Credibility scores](#4-credibility-scores)
5. [Xref resolution helpers](#5-xref-resolution)
6. [TxData / legacy TxGNN mapping](#6-legacy-mapping)
7. [End-to-end pipeline sketch](#7-pipeline)


In [1]:
import sys
sys.path.insert(0, '..')   # repo root

from manage_db.kg_schema import (
    NodeType, NODE_TYPES, NODE_XREF_COLUMNS,
    Relation, RelationKind, RELATIONS, RELATION_BY_NAME,
    RELATIONS_BY_SOURCE, RELATIONS_BY_TARGET,
    Credibility,
    XrefResolution, XREF_RESOLUTION, XREF_BY_COLUMN,
    TXDATA_NODE_TYPE_MAP, TXDATA_RELATION_MAP, TXDATA_RELATION_FLIP,
    EDGE_PARQUET_COLUMNS, NODE_PARQUET_PRIMARY_COLUMN,
    relation_names, node_type_names, relations_between,
    xref_columns_for, primary_ontology_for,
)
import pandas as pd

---
## 1 — Node types

Each node type has exactly **one** primary ontology.  All node Parquet files
store the primary ID in a column called `id`.

In [2]:
rows = []
for nt, info in NODE_TYPES.items():
    rows.append({
        "node_type": nt.value,
        "primary_ontology": info.primary_ontology,
        "id_format": info.id_format,
        "bionty_registry": info.bionty_registry or "—",
        "example_id": info.example_id,
    })

pd.DataFrame(rows).set_index("node_type")

,primary_ontology,id_format,bionty_registry,example_id
node_type,,,,
gene,Ensembl,ENSG<11digits>,bionty.Gene,ENSG00000139618
transcript,Ensembl,ENST<11digits>,—,ENST00000380152
protein,Ensembl Protein,ENSP[0-9]{11},—,ENSP00000369497
disease,EFO,EFO:<7digits>,bionty.Disease,EFO:0000305
cell_type,CL,CL:<7digits>,bionty.CellType,CL:0000576
tissue,UBERON,UBERON:<7digits>,bionty.Tissue,UBERON:0002107
molecule,ChEMBL,CHEMBL<int>,pertdb.Compound,CHEMBL941
phenotype,HP,HP:<7digits>,bionty.Phenotype,HP:0000118
pathway,Reactome,R-HSA-<int>,bionty.Pathway,R-HSA-5633007


In [3]:
# Convenience helpers
print("All node type names:", node_type_names())
print("Primary ontology for disease:", primary_ontology_for(NodeType.DISEASE))
print("Primary ontology for gene:   ", primary_ontology_for(NodeType.GENE))

All node type names: ['paper', 'gene', 'transcript', 'protein', 'pathway', 'molecule', 'mutation', 'disease', 'cell_type', 'tissue', 'phenotype', 'cell_line', 'organism', 'dataset', 'enhancer']
Primary ontology for disease: EFO
Primary ontology for gene:    Ensembl


---
## 2 — Cross-reference columns

Every node Parquet file contains `id` (primary) plus the `xref_columns` for
that node type.  These columns are nullable strings.

**Example — Disease node columns:**
```
id            EFO:0000305          ← primary (always present)
mondo_id      MONDO:0007254        ← cross-reference (nullable)
omim_id       114480               ← cross-reference (nullable)
doid_id       DOID:1612            ← cross-reference (nullable)
icd10_code    C50.9                ← cross-reference (nullable)
mesh_id       D001943              ← cross-reference (nullable)
hp_id         (null)
```

In [4]:
# Show all xref columns per node type
for nt in NodeType:
    cols = xref_columns_for(nt)
    if cols:
        print(f"{nt.value:12s}  →  {', '.join(cols)}")
    else:
        print(f"{nt.value:12s}  →  (no xrefs)")

paper         →  doi, pmc_id, arxiv_id
gene          →  ncbi_gene_id, hgnc_id, uniprot_id, gene_name
transcript    →  ensembl_gene_id, protein_id, refseq_mrna, ccds_id
protein       →  ensembl_gene_id, uniprot_id, refseq_protein, pdb_ids
pathway       →  go_id, kegg_id
molecule      →  drugbank_id, pubchem_cid, cas_rn, inchikey, smiles
mutation      →  hgvs, clinvar_id, gnomad_id
disease       →  mondo_id, omim_id, doid_id, icd10_code, mesh_id, hp_id
cell_type     →  uberon_id, mesh_id
tissue        →  bto_id, mesh_id, fma_id
phenotype     →  mondo_id, efo_id, mp_id, mesh_id
cell_line     →  ccle_name, cosmic_id, efo_id
organism      →  gbif_id
dataset       →  (no xrefs)
enhancer      →  ensembl_regulatory_id, encode_experiment_id


In [5]:
# Example: build the full column list for a gene Parquet file
gene_parquet_columns = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(NodeType.GENE))
print("Gene Parquet required columns:", gene_parquet_columns)

Gene Parquet required columns: ['id', 'ncbi_gene_id', 'hgnc_id', 'uniprot_id', 'gene_name']


### Simulated node record

Below is what a row in `data/kg/nodes/disease.parquet` looks like:

In [6]:
import pandas as pd

disease_columns = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(NodeType.DISEASE)) + ["name"]
example_disease = pd.DataFrame([{
    "id":         "EFO:0000305",
    "mondo_id":   "MONDO:0007254",
    "omim_id":    "114480",
    "doid_id":    "DOID:1612",
    "icd10_code": "C50.9",
    "mesh_id":    "D001943",
    "hp_id":      None,
    "name":       "breast carcinoma",
}], columns=disease_columns)

example_disease

,id,mondo_id,omim_id,doid_id,icd10_code,mesh_id,hp_id,name
0,EFO:0000305,MONDO:0007254,114480,DOID:1612,C50.9,D001943,None,breast carcinoma


---
## 3 — Relation taxonomy

76 canonical directed relation types grouped by `RelationKind`.

In [7]:
# Summary by kind
from collections import Counter

kind_counts = Counter(r.kind.value for r in RELATIONS)
pd.Series(kind_counts, name="count").sort_values(ascending=False)

genetic            9
pharmacological    8
ontological        8
experimental       7
metadata           7
regulatory         5
expression         5
disease_assoc      5
phenotype_assoc    3
physical           3
pathway            3
central_dogma      2
epidemiological    1
literature         1
Name: count, dtype: int64

In [8]:
# All relation names
print("Total relations:", len(RELATIONS))
for name in relation_names():
    print(" ", name)

Total relations: 67
  gene_has_transcript
  transcript_encodes_protein
  mutation_in_gene
  mutation_associated_gene
  mutation_affects_transcript
  mutation_causes_protein_change
  mutation_overlaps_enhancer
  mutation_associated_disease
  mutation_associated_phenotype
  gene_associated_phenotype
  mutation_affects_molecule_response
  gene_ortholog_gene
  enhancer_regulates_gene
  enhancer_regulates_transcript
  gene_coexpressed_gene
  tissue_expresses_gene
  tissue_expresses_protein
  cell_type_expresses_gene
  cell_type_expresses_protein
  cell_line_expresses_gene
  cell_line_expresses_protein
  cell_line_gene_essentiality
  gene_interacts_gene
  tf_regulates_gene
  tf_binds_enhancer
  transcript_interacts_protein
  transcript_interacts_gene
  protein_interacts_protein
  pathway_contains_gene
  pathway_contains_protein
  pathway_child_of_pathway
  molecule_in_pathway
  molecule_targets_gene
  molecule_targets_protein
  molecule_treats_disease
  molecule_contraindicates_disease
  mol

In [9]:
# Look up a relation by name
rel = RELATION_BY_NAME["molecule_treats_disease"]
print(f"name:   {rel.name}")
print(f"source: {rel.source.value}")
print(f"target: {rel.target.value}")
print(f"kind:   {rel.kind.value}")
print(f"direct: {rel.direct}")
print(f"notes:  {rel.notes}")

name:   molecule_treats_disease
source: molecule
target: disease
kind:   pharmacological
direct: False
notes:  Indication (clinical)


In [10]:
# Relations between two specific node types
rels = relations_between(NodeType.DISEASE, NodeType.GENE)
print("Disease → Gene relations:")
for r in rels:
    print(f"  {r.name:40s}  direct={r.direct}  kind={r.kind.value}")

Disease → Gene relations:


In [11]:
# All relations that target Disease
print("Relations targeting Disease:")
for r in RELATIONS_BY_TARGET.get(NodeType.DISEASE, []):
    print(f"  {r.source.value:12s} → disease   [{r.name}]")

Relations targeting Disease:
  mutation     → disease   [mutation_associated_disease]
  molecule     → disease   [molecule_treats_disease]
  molecule     → disease   [molecule_contraindicates_disease]
  gene         → disease   [disease_associated_gene]
  protein      → disease   [disease_associated_protein]
  pathway      → disease   [disease_involves_pathway]
  disease      → disease   [disease_subtype_of_disease]
  disease      → disease   [disease_comorbid_disease]
  cell_type    → disease   [cell_type_involved_in_disease]
  cell_line    → disease   [cell_line_models_disease]
  dataset      → disease   [dataset_contains_disease]


In [12]:
# Direct vs indirect relations
direct = [r for r in RELATIONS if r.direct]
indirect = [r for r in RELATIONS if not r.direct]
print(f"Direct:   {len(direct)}")
print(f"Indirect: {len(indirect)}")

Direct:   44
Indirect: 23


---
## 4 — Credibility scores

Every edge in a Parquet file carries a `credibility` column (int 1–3).

In [13]:
for c in Credibility:
    print(f"{c.value}  {c.name:20s}  — {c.__doc__ or ''}")

# Usage: assign credibility when ingesting edges
print("\nExample:")
print("  ChEMBL indication  →", int(Credibility.ESTABLISHED_FACT))
print("  Single GWAS hit    →", int(Credibility.SINGLE_EVIDENCE))

1  SINGLE_EVIDENCE       — Evidence credibility level for edges.
2  MULTI_EVIDENCE        — Evidence credibility level for edges.
3  ESTABLISHED_FACT      — Evidence credibility level for edges.

Example:
  ChEMBL indication  → 3
  Single GWAS hit    → 1


### Edge Parquet schema

Every edge file has at minimum these columns:

In [14]:
pd.DataFrame(EDGE_PARQUET_COLUMNS, columns=["column", "description"])

,column,description
0,x_id,str — primary ontology ID of the source node
1,x_type,str — NodeType value of the source node
2,y_id,str — primary ontology ID of the target node
3,y_type,str — NodeType value of the target node
4,relation,str — canonical Relation.name
5,display_relation,str — human-readable label
6,source,str — database / dataset the edge came from
7,credibility,int — 1 | 2 | 3 (Credibility enum value)


In [15]:
# Example edge row
example_edge = pd.DataFrame([{
    "x_id":             "CHEMBL941",
    "x_type":           "molecule",
    "y_id":             "EFO:0000305",
    "y_type":           "disease",
    "relation":         "molecule_treats_disease",
    "display_relation": "treats",
    "source":           "ChEMBL",
    "credibility":      3,
}])
example_edge

,x_id,x_type,y_id,y_type,relation,display_relation,source,credibility
0,CHEMBL941,molecule,EFO:0000305,disease,molecule_treats_disease,treats,ChEMBL,3


---
## 5 — Xref resolution helpers

`XREF_RESOLUTION` describes how to go from an external ID *to* the canonical
primary ID.  Each entry maps one `xref_column` to a resolution recipe
(description + optional API URL template).

In [16]:
# All xref resolutions for Disease
disease_xrefs = [x for x in XREF_RESOLUTION if x.node_type == NodeType.DISEASE]
for x in disease_xrefs:
    print(f"  {x.xref_column:15s}  ({x.from_namespace})")
    print(f"    → {x.description}")
    if x.url_template:
        print(f"    URL: {x.url_template}")

  mondo_id         (MONDO)
    → MONDO ID → EFO via OXO cross-reference service
    URL: https://www.ebi.ac.uk/spot/oxo/api/mappings?fromId={id}&toDb=EFO
  omim_id          (OMIM)
    → OMIM MIM number → EFO via MONDO owl mappings (skos:exactMatch)
  doid_id          (DOID)
    → Disease Ontology ID → EFO via OXO
    URL: https://www.ebi.ac.uk/spot/oxo/api/mappings?fromId={id}&toDb=EFO
  icd10_code       (ICD-10)
    → ICD-10 code → EFO via EFO annotations or MONDO mappings
  mesh_id          (MeSH)
    → MeSH disease descriptor → EFO via UMLS / OXO mappings
    URL: https://www.ebi.ac.uk/spot/oxo/api/mappings?fromId={id}&toDb=EFO


In [17]:
# Look up a specific xref resolution
res = XREF_BY_COLUMN[("drugbank_id", NodeType.MOLECULE)]
print("DrugBank → ChEMBL resolution:")
print(f"  from_namespace: {res.from_namespace}")
print(f"  description:    {res.description}")
print(f"  URL template:   {res.url_template}")

# Build an actual URL for DB01267 (Imatinib / Gleevec)
drugbank_id = "DB01267"
url = res.url_template.format(id=drugbank_id)
print(f"  → Resolved URL: {url}")

DrugBank → ChEMBL resolution:
  from_namespace: DrugBank
  description:    DrugBank accession → ChEMBL via UniChem
  URL template:   https://www.ebi.ac.uk/unichem/api/v1/compounds?sourceId={id}&sourceName=drugbank
  → Resolved URL: https://www.ebi.ac.uk/unichem/api/v1/compounds?sourceId=DB01267&sourceName=drugbank


In [18]:
# Summary table of all xref resolutions
xref_rows = [{
    "node_type": x.node_type.value,
    "xref_column": x.xref_column,
    "from_namespace": x.from_namespace,
    "has_api": bool(x.url_template),
} for x in XREF_RESOLUTION]
pd.DataFrame(xref_rows)

,node_type,xref_column,from_namespace,has_api
0,gene,ncbi_gene_id,NCBI Gene ID,True
1,gene,hgnc_id,HGNC,True
2,gene,uniprot_id,UniProt,True
3,protein,ensembl_gene_id,Ensembl Gene ID,True
4,protein,refseq_protein,RefSeq Protein,True
5,disease,mondo_id,MONDO,True
6,disease,omim_id,OMIM,False
7,disease,doid_id,DOID,True
8,disease,icd10_code,ICD-10,False
9,disease,mesh_id,MeSH,True


---
## 6 — TxData / legacy TxGNN mapping

TxData / legacy TxGNN exports use different node type strings and relation names.
`TXDATA_NODE_TYPE_MAP` and `TXDATA_RELATION_MAP` translate them to the active Jouvence schema.

In [19]:
# Node type mapping
pd.DataFrame(
    [(k, v.value) for k, v in TXDATA_NODE_TYPE_MAP.items()],
    columns=["legacy_type", "new_type"]
)

,legacy_type,new_type
0,gene/protein,gene
1,drug,molecule
2,disease,disease
3,effect/phenotype,phenotype
4,anatomy,tissue
5,biological_process,pathway
6,molecular_function,pathway
7,cellular_component,pathway
8,pathway,pathway
9,exposure,molecule


In [20]:
# Relation mapping
pd.DataFrame(
    [(k, v) for k, v in TXDATA_RELATION_MAP.items()],
    columns=["legacy_relation", "canonical_relation"]
)

,legacy_relation,canonical_relation
0,indication,molecule_treats_disease
1,contraindication,molecule_contraindicates_disease
2,off-label use,molecule_treats_disease
3,target,molecule_targets_gene
4,enzyme,molecule_targets_gene
5,transporter,molecule_targets_gene
6,carrier,molecule_targets_gene
7,biomarker,disease_associated_gene
8,disease_protein,disease_associated_gene
9,protein_protein,gene_interacts_gene


In [21]:
# Relations whose source/target must be flipped during migration
print("Relations that require x/y swap on migration:")
for r in sorted(TXDATA_RELATION_FLIP):
    canonical = TXDATA_RELATION_MAP.get(r, "(not in map)")
    print(f"  {r:35s}  →  {canonical}")

Relations that require x/y swap on migration:
  anatomy_protein_absent               →  tissue_expresses_gene
  anatomy_protein_present              →  tissue_expresses_gene
  bioprocess_protein                   →  pathway_contains_gene
  cellcomp_protein                     →  pathway_contains_gene
  disease_protein                      →  disease_associated_gene
  molfunc_protein                      →  pathway_contains_gene
  pathway_protein                      →  pathway_contains_gene


---
## 7 — End-to-end pipeline sketch

Below is a minimal template showing how to go from a raw edge CSV (using legacy
TxGNN naming) to a compliant edge Parquet file with canonical IDs.

In [22]:
import pandas as pd
from manage_db.kg_schema import (
    TXDATA_NODE_TYPE_MAP, TXDATA_RELATION_MAP, TXDATA_RELATION_FLIP,
    RELATION_BY_NAME, Credibility, NODE_PARQUET_PRIMARY_COLUMN,
)

# ── Simulated raw TxGNN edge data ────────────────────────────────────────
raw_edges = pd.DataFrame([
    # x_id       x_type         y_id                y_type     relation
    ("imatinib", "drug",        "chronic myeloid leukemia", "disease",  "indication"),
    ("BRCA2",    "gene/protein","breast cancer",    "disease",  "biomarker"),
    ("BRCA2",    "gene/protein","DNA repair",       "biological_process", "bioprocess_protein"),
], columns=["x_id", "x_type", "y_id", "y_type", "relation"])

print("Raw edges:")
display(raw_edges)

# ── Step 1: translate node types ─────────────────────────────────────────
raw_edges["x_type"] = raw_edges["x_type"].map(TXDATA_NODE_TYPE_MAP).apply(lambda x: getattr(x, "value", x) if x else None)
raw_edges["y_type"] = raw_edges["y_type"].map(TXDATA_NODE_TYPE_MAP).apply(lambda x: getattr(x, "value", x) if x else None)

# ── Step 2: flip edges that have reversed direction in legacy data ────────
flip_mask = raw_edges["relation"].isin(TXDATA_RELATION_FLIP)
raw_edges.loc[flip_mask, ["x_id", "x_type", "y_id", "y_type"]] = (
    raw_edges.loc[flip_mask, ["y_id", "y_type", "x_id", "x_type"]].values
)

# ── Step 3: translate relation names ─────────────────────────────────────
raw_edges["relation"] = raw_edges["relation"].map(TXDATA_RELATION_MAP)

# ── Step 4: add required Parquet columns ─────────────────────────────────
raw_edges["display_relation"] = raw_edges["relation"].apply(
    lambda r: RELATION_BY_NAME[r].notes if r in RELATION_BY_NAME else r
)
raw_edges["source"] = "TxGNN-legacy"
raw_edges["credibility"] = int(Credibility.ESTABLISHED_FACT)

print("\nNormalised edges (ready to write as Parquet):")
display(raw_edges)

Raw edges:


,x_id,x_type,y_id,y_type,relation
0,imatinib,drug,chronic myeloid leukemia,disease,indication
1,BRCA2,gene/protein,breast cancer,disease,biomarker
2,BRCA2,gene/protein,DNA repair,biological_process,bioprocess_protein



Normalised edges (ready to write as Parquet):


,x_id,x_type,y_id,y_type,relation,display_relation,source,credibility
0,imatinib,molecule,chronic myeloid leukemia,disease,molecule_treats_disease,Indication (clinical),TxGNN-legacy,3
1,BRCA2,gene,breast cancer,disease,disease_associated_gene,Gene→disease direction for causal/directed dis...,TxGNN-legacy,3
2,DNA repair,pathway,BRCA2,gene,pathway_contains_gene,Reactome / GO,TxGNN-legacy,3


In [23]:
# ── Validating a node DataFrame against schema ────────────────────────────
from manage_db.kg_schema import xref_columns_for, NodeType

def validate_node_df(df: pd.DataFrame, node_type: NodeType) -> list[str]:
    """Return a list of schema violations for a node DataFrame."""
    errors = []
    required = [NODE_PARQUET_PRIMARY_COLUMN] + list(xref_columns_for(node_type))
    for col in required:
        if col not in df.columns:
            errors.append(f"Missing required column: '{col}'")
    if NODE_PARQUET_PRIMARY_COLUMN in df.columns and df[NODE_PARQUET_PRIMARY_COLUMN].isna().any():
        errors.append("Primary ID column 'id' contains null values")
    return errors


# Example: well-formed gene DataFrame
good_genes = pd.DataFrame([{
    "id":           "ENSG00000139618",
    "ncbi_gene_id": "675",
    "hgnc_id":      "HGNC:1101",
    "uniprot_id":   "P51587",
    "gene_name":    "BRCA2",
}])

errors = validate_node_df(good_genes, NodeType.GENE)
print("Validation errors (good df):", errors or "none ✓")

# Example: missing primary ID
bad_genes = good_genes.drop(columns=["id"])
errors = validate_node_df(bad_genes, NodeType.GENE)
print("Validation errors (bad df): ", errors)

Validation errors (good df): none ✓
Validation errors (bad df):  ["Missing required column: 'id'"]


---
## Quick-reference cheat sheet

| Task | Code |
|------|------|
| Primary ontology for a node type | `primary_ontology_for(NodeType.DISEASE)` |
| Xref columns for a node type | `xref_columns_for(NodeType.MOLECULE)` |
| All relations from source | `RELATIONS_BY_SOURCE[NodeType.MOLECULE]` |
| Relations between two types | `relations_between(NodeType.DISEASE, NodeType.GENE)` |
| Lookup relation by name | `RELATION_BY_NAME["molecule_treats_disease"]` |
| Xref resolution recipe | `XREF_BY_COLUMN[("drugbank_id", NodeType.MOLECULE)]` |
| Translate TxData node type | `TXDATA_NODE_TYPE_MAP["gene/protein"]` |
| Translate TxData relation | `TXDATA_RELATION_MAP["indication"]` |
| Credibility for curated DB | `int(Credibility.ESTABLISHED_FACT)` → 3 |